# Feature Set: behavior

**관점**: '어떻게 소비하는가' - 시간/금액/할부/점포/수입 행동 통계

이 노트북은 **하나의 feature set** 에 대해 5-fold 외부 CV로 AutoGluon(블랙박스 모델)을 학습하고, CV 지표(AUC/LogLoss/Brier)를 확인한 뒤 **OOF 예측**과 **test 예측**을 저장합니다.

- `oof_{setname}.csv` : train OOF (메타모델 학습 입력)
- `oof_test_{setname}.csv` : test 예측 평균 (메타모델 추론 입력)

> 모든 set 노트북은 동일한 `folds.csv` split 을 공유합니다 (스태킹 정합성).

## 0. 설치 & 라이브러리

In [28]:
import sys
!{sys.executable} -m pip install ipython-autotime
!{sys.executable} -m pip install autogluon.tabular gensim

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
time: 3.87 s (started: 2026-06-06 20:10:15 +09:00)



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
# 로그에서 제안하는 대로 버전을 맞춰 설치
!pip install "autogluon.tabular[xgboost,fastai]==1.5.0"

Defaulting to user installation because normal site-packages is not writeable
time: 2.45 s (started: 2026-06-06 20:10:18 +09:00)



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
%load_ext autotime

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from autogluon.tabular import TabularPredictor
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('autogluon').setLevel(logging.ERROR)

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# ---------------- 전역 설정 ----------------
SEED        = 42
N_SPLITS    = 5
TIME_LIMIT  = 3600          # 1800
PRESETS     = 'best_quality'
EVAL_METRIC = 'roc_auc'    # AutoGluon 내부 모델 선택 기준

DATA_DIR = '../../data/raw'
OOF_DIR  = '../../outputs/oof'
os.makedirs(OOF_DIR, exist_ok=True)

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 4.4 ms (started: 2026-06-06 20:10:21 +09:00)


## 1. 데이터 로드

In [31]:
# 거래 단위 원본 데이터 로드
X_train_trans = pd.read_csv("competition_data/train_transactions.csv", encoding='cp949')
X_test_trans  = pd.read_csv("competition_data/test_transactions.csv",  encoding='cp949')
y_train       = pd.read_csv('competition_data/y_train.csv',            encoding='cp949')

print('train trans:', X_train_trans.shape, '| test trans:', X_test_trans.shape)
print('y_train:', y_train.shape)

train trans: (1036653, 16) | test trans: (689777, 16)
y_train: (30000, 2)
time: 5.19 s (started: 2026-06-06 20:10:21 +09:00)


## 2. Fold split 로드/생성 (공유)

In [32]:
# ===== Fold split: 모든 feature set / 노트북이 공유 =====
# 첫 실행에서 folds.csv 가 없으면 생성, 있으면 로드 -> 7개 set 모두 동일 split 사용 보장
FOLDS_PATH = f'{OOF_DIR}/folds.csv'

if os.path.exists(FOLDS_PATH):
    folds_df = pd.read_csv(FOLDS_PATH)
    print('기존 folds.csv 로드:', folds_df.shape)
else:
    base = y_train[['custid', 'gender']].sort_values('custid').reset_index(drop=True)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    base['fold'] = -1
    for f, (_, val_idx) in enumerate(skf.split(base['custid'], base['gender'])):
        base.loc[val_idx, 'fold'] = f
    folds_df = base[['custid', 'fold']]
    folds_df.to_csv(FOLDS_PATH, index=False)
    print('folds.csv 신규 생성:', folds_df.shape)

print(folds_df['fold'].value_counts().sort_index())

기존 folds.csv 로드: (30000, 2)
fold
0    6000
1    6000
2    6000
3    6000
4    6000
Name: count, dtype: int64
time: 15.9 ms (started: 2026-06-06 20:10:26 +09:00)


## 3. Feature 생성  ← 팀원이 통째로 교체하는 영역
`features_train`, `features_test` 두 변수를 `custid` + feature 컬럼 형태로 만들면 됩니다.

In [33]:
SETNAME = 'behavior'

time: 6.4 ms (started: 2026-06-06 20:10:26 +09:00)


In [34]:
# ===== FEATURE 생성: 고객 단위 내부 피처 =====
# 관점: 고객(custid) 내부에서만 계산하는 누수 없는 소비/점포/상품 행동 피처
#  - 소비 패턴      : tot_amt, dis_amt, net_amt, inst_mon, inst_fee
#  - 점포/상품 패턴 : str_nm, goodcd, import_flg

KEY = 'custid'

def make_features(raw):
    use_cols = [KEY,
                'tot_amt', 'dis_amt', 'net_amt', 'inst_mon', 'inst_fee',   # 소비 패턴
                'str_nm', 'goodcd', 'import_flg']                          # 점포/상품 패턴
    df = raw[use_cols].copy()
    g  = df.groupby(KEY)

    # 환불 기준 금액: net_amt 우선, net_amt가 0이거나 결측이면 tot_amt 대체(1등 코드 참고)
    df['refund_base'] = df['net_amt'].where(df['net_amt'].notna() & (df['net_amt'] != 0),
                                            df['tot_amt'])

    # ----------------------------------------------------------------
    # 1. 소비 패턴
    # ----------------------------------------------------------------
    # 1-1. 환불 강도
    df['is_refund']    = (df['refund_base'] < 0).astype(int)
    df['refund_amt']   = df['refund_base'].where(df['refund_base'] < 0, 0).abs()  # 환불액(양수)
    df['purchase_amt'] = df['refund_base'].where(df['refund_base'] >= 0, 0)  # 구매액

    refund = pd.DataFrame({
        'refund_count'    : g['is_refund'].sum(),  # 환불 횟수
        'refund_ratio'    : g['is_refund'].mean(), # 총구매 횟수 당 환불 횟수 비율
        'refund_amt_sum'  : g['refund_amt'].sum(),  # 환불 총액
        'refund_amt_mean' : df[df['is_refund'] == 1].groupby(KEY)['refund_amt'].mean(),  # 건당 평균 환불액 (평균적으로 얼마나 비싼걸 환불했나 확인)
        'purchase_amt_sum': g['purchase_amt'].sum(),  # 구매 총액
    })
    refund['refund_flag']      = (refund['refund_count'] > 0).astype(int) # 환불 경험 여부
    refund['net_revenue']      = refund['purchase_amt_sum'] - refund['refund_amt_sum'] # 실매출 (이렇게 나눠놔야 group by 했을때 어떤 환불 이력을 바탕으로 실매출이 나왔는지 알 수 있음)
    refund['refund_amt_ratio'] = refund['refund_amt_sum'] / refund['purchase_amt_sum'].replace(0, np.nan) # 금액 기준 환불 강도
    refund = refund.fillna(0)

    # 1-2. 금액 분포 + 분위수 + 고가 거래 빈도
    amount = g.agg(
        tx_cnt       = ('tot_amt', 'size'), #거래 건수
        tot_amt_sum  = ('tot_amt', 'sum'),
        tot_amt_mean = ('tot_amt', 'mean'),
        tot_amt_std  = ('tot_amt', 'std'), #거래액 표준편차
        tot_amt_max  = ('tot_amt', 'max'),
        tot_amt_min  = ('tot_amt', 'min'),
        tot_amt_med  = ('tot_amt', 'median'),
        net_amt_sum  = ('net_amt', 'sum'),
        net_amt_mean = ('net_amt', 'mean'),
    )
    amount['tot_amt_p95']  = g['tot_amt'].quantile(0.95) # 고객 내 95분위 거래액 (이상치 확인)
    amount['tot_amt_p99']  = g['tot_amt'].quantile(0.99) # 고객 내 99분위 거래액 (이상치 확인)
    amount['tot_amt_iqr']  = g['tot_amt'].quantile(0.75) - g['tot_amt'].quantile(0.25) # 사분위 범위
    amount['tot_amt_skew'] = g['tot_amt'].skew() # 금액 분포 왜도(고가 쏠림) #0:정규 분포, 0> : 오른쪽(큰값) 치우침
    amount['tot_amt_cv']        = amount['tot_amt_std'] / amount['tot_amt_mean'].abs().replace(0, np.nan) # 변동계수(고객의 소비(tot_amt)가 얼마나 들쭉날쭉)
    amount['tot_amt_max_ratio'] = amount['tot_amt_max'] / amount['tot_amt_sum'].replace(0, np.nan) # 최고가 비중(1에 가까울수록 한탕형(목적성 소비))

    cust_mean = g['tot_amt'].transform('mean')
    df['is_high_tx'] = (df['tot_amt'] > cust_mean * 3).astype(int) #개별 거래 금액이 본인 평균 구매 금액의 3배 초과하면 1 or 0
    amount['high_tx_count'] = df.groupby(KEY)['is_high_tx'].sum() # 고가 거래 건수 (평소 대비 무리 지출 횟수)
    amount['high_tx_ratio'] = df.groupby(KEY)['is_high_tx'].mean() # 고가 거래 비율 (전체 방문/거래 횟수 중 평소보다 크게 지출한 거래의 비율)
    amount = amount.fillna(0)

    # 1-3. 할인
    df['has_dis']  = (df['dis_amt'] != 0).astype(int)
    df['dis_rate'] = df['dis_amt'] / df['tot_amt'].replace(0, np.nan)
    discount = g.agg(
        dis_amt_sum   = ('dis_amt', 'sum'),
        dis_amt_mean  = ('dis_amt', 'mean'),
        dis_use_ratio = ('has_dis', 'mean'), # 할인 적용 거래 비율
        dis_rate_mean = ('dis_rate', 'mean'), # 평균 할인율
        dis_rate_max  = ('dis_rate', 'max'),
    )
    discount['dis_intensity'] = g['dis_amt'].sum() / g['tot_amt'].sum().replace(0, np.nan) # 누적 할인 강도
    discount = discount.fillna(0)

    # 1-4. 할부
    df['is_installment'] = (df['inst_mon'] > 1).astype(int)
    df['is_lump']        = (df['inst_mon'] <= 1).astype(int)
    df['is_no_interest'] = (df['inst_fee'] == 1).astype(int)
    df['is_long_inst']   = (df['inst_mon'] >= 6).astype(int)
    install = g.agg(
        inst_use_ratio    = ('is_installment', 'mean'),
        lump_ratio        = ('is_lump', 'mean'),
        inst_mon_mean     = ('inst_mon', 'mean'),
        inst_mon_max      = ('inst_mon', 'max'),
        long_inst_ratio   = ('is_long_inst', 'mean'),
        no_interest_ratio = ('is_no_interest', 'mean'),
    )
    install['inst_mon_when_used'] = df[df['is_installment'] == 1].groupby(KEY)['inst_mon'].mean()
    install = install.fillna(0)

    # ----------------------------------------------------------------
    # 2. 점포/상품 패턴
    # ----------------------------------------------------------------
    # 2-1. 점포 (요약 통계만 사용: 점포명 원-핫은 train/test 불일치 → 제외)
    store = pd.DataFrame(index=df[KEY].unique()).sort_index(); store.index.name = KEY
    store['n_stores'] = g['str_nm'].nunique() # 방문 점포 수
    store_ct    = pd.crosstab(df[KEY], df['str_nm'])
    store_share = store_ct.div(store_ct.sum(axis=1), axis=0)
    store['top_store_share'] = store_share.max(axis=1)  # 주이용 점포 집중도
    store['store_hhi']       = (store_share ** 2).sum(axis=1)  # 점포 충성도(HHI)
    store = store.fillna(0)

    # 2-2. 상품 (goodcd)
    good = pd.DataFrame(index=df[KEY].unique()).sort_index(); good.index.name = KEY
    good['n_goods']        = g['goodcd'].nunique()  # 구매 상품종류 수
    good['goods_per_tx']   = good['n_goods'] / g['goodcd'].size()  # 다양성
    good['repeat_ratio']   = 1 - good['goods_per_tx'] # 재구매 성향
    good['top_good_share'] = g['goodcd'].agg(lambda s: s.value_counts(normalize=True).iloc[0])
    good = good.fillna(0)

    # 2-3. 수입품 (import_flg)
    imp = g.agg(import_cnt=('import_flg', 'sum'), import_ratio=('import_flg', 'mean'))
    imp['import_flag'] = (imp['import_cnt'] > 0).astype(int) # 수입품 구매 경험

    # ----------------------------------------------------------------
    # 3. 추가 파생 피처 (고객 내부 계산 → 누수 없음)
    # ----------------------------------------------------------------
    
    # 3-1. 절대 고액 결제 (충동구매): 고객 상대 기준(high_tx_*)과 상호보완되는 절대 임계
    df['is_big_50']  = (df['tot_amt'] > 500000).astype(int) # 50만원 초과
    df['is_big_100'] = (df['tot_amt'] > 1000000).astype(int) # 100만원 초과
    extra = pd.DataFrame(index=df[KEY].unique()).sort_index(); extra.index.name = KEY
    extra['big50_count']  = df.groupby(KEY)['is_big_50'].sum() # 50만원 초과 건수
    extra['big50_ratio']  = df.groupby(KEY)['is_big_50'].mean()  # 50만원 초과 비율
    extra['big100_count'] = df.groupby(KEY)['is_big_100'].sum() # 100만원 초과 건수
    extra['big100_ratio'] = df.groupby(KEY)['is_big_100'].mean() # 100만원 초과 비율

    # 3-2. 고액 x 결제수단: 충동성(고액 일시불) vs 계획성(고액 장기할부)
    df['big_lump']      = (df['is_big_50'] & (df['inst_mon'] <= 1)).astype(int) # 고액 일시불
    df['big_long_inst'] = (df['is_big_50'] & (df['inst_mon'] >= 6)).astype(int) # 고액 장기할부
    extra['big_lump_ratio']      = df.groupby(KEY)['big_lump'].mean() # 고액 일시불 비율(충동성)
    extra['big_long_inst_ratio'] = df.groupby(KEY)['big_long_inst'].mean() # 고액 장기할부 비율(계획성)

    # 3-3. 수입품 객단가/금액 비중 (양수 구매액 기준 → 환불로 인한 왜곡 방지)
    df['imp_purchase'] = (df['import_flg'] * df['purchase_amt']) # 수입품 구매액(환불 제외)
    imp_amt_sum = df.groupby(KEY)['imp_purchase'].sum()
    pur_amt_sum = df.groupby(KEY)['purchase_amt'].sum().replace(0, np.nan)
    extra['import_amt_share'] = (imp_amt_sum / pur_amt_sum).fillna(0) # 소비액 중 수입품 비중
    imp_mean = df[df['import_flg'] == 1].groupby(KEY)['tot_amt'].mean()
    extra['import_tot_amt_mean'] = imp_mean # 수입품 평균 객단가
    extra['import_tot_amt_mean'] = extra['import_tot_amt_mean'].fillna(0)

    # 3-4. 점포별 객단가 편차: 점포마다 쓰는 금액 차이(프리미엄 점포 선호 신호)
    #      단일 점포 고객은 편차 정의 불가 → 0 처리
    store_mean = df.groupby([KEY, 'str_nm'])['tot_amt'].mean()
    extra['store_price_std'] = store_mean.groupby(KEY).std().fillna(0)

    # 3-5. 무할인 정가 구매 성향 (가격 비민감 고객)
    df['no_dis'] = (df['dis_amt'] == 0).astype(int)
    extra['no_discount_ratio'] = df.groupby(KEY)['no_dis'].mean() # 할인 0원 거래 비율
    
    # 3-6. 환불 집중도: 최대 환불 1건이 전체 환불액에서 차지하는 비중
    #      대형 단발 환불(1.0에 근접) vs 잦은 소액 환불(0에 근접), 무환불 고객은 0
    r_max = df.groupby(KEY)['refund_amt'].max()
    r_sum = df.groupby(KEY)['refund_amt'].sum().replace(0, np.nan)
    extra['refund_concentration'] = (r_max / r_sum).fillna(0)
    extra = extra.fillna(0)

    # --- 통합 ---
    feats = (refund.join(amount).join(discount).join(install)
             .join(store).join(good).join(imp).join(extra))
    feats = feats.fillna(0).reset_index()      # custid 를 컬럼으로
    return feats

features_train = make_features(X_train_trans)
features_test  = make_features(X_test_trans)
print('고객 단위 피처:', features_train.shape[1] - 1)

# ===== train/test feature 컬럼 정렬 (카테고리/통계 불일치 방지) =====
feat_cols_ref = [c for c in features_train.columns if c != 'custid']
features_test = features_test.reindex(columns=['custid'] + feat_cols_ref, fill_value=0)
print('정렬 후 feature 수:', len(feat_cols_ref))

고객 단위 피처: 59
정렬 후 feature 수: 59
time: 19 s (started: 2026-06-06 20:10:26 +09:00)


## 4. 5-fold 외부 CV (AutoGluon = 하나의 블랙박스 모델)

In [35]:
# ===== 5-fold 외부 CV: AutoGluon 을 하나의 블랙박스 모델로 취급 =====
# - 각 fold: train fold 로 AutoGluon fit -> valid fold 예측 = OOF (leakage 없음)
# - test: 5개 fold 모델 예측의 평균 = test meta feature
SETNAME = SETNAME  # 위에서 정의됨

# custid / target / fold 정렬 맞추기
data = features_train.merge(folds_df, on='custid', how='inner') \
                     .merge(y_train[['custid', 'gender']], on='custid', how='inner')
data = data.sort_values('custid').reset_index(drop=True)

feat_cols = [c for c in features_train.columns if c != 'custid']

oof_pred  = np.zeros(len(data))
test_pred = np.zeros(len(features_test))
test_sorted = features_test.sort_values('custid').reset_index(drop=True)

for f in range(N_SPLITS):
    tr_idx = data.index[data['fold'] != f]
    va_idx = data.index[data['fold'] == f]

    train_part = data.loc[tr_idx, feat_cols + ['gender']]
    valid_part = data.loc[va_idx, feat_cols]

    predictor = TabularPredictor(
        label='gender', problem_type='binary',
        eval_metric=EVAL_METRIC, verbosity=0,
    ).fit(train_data=train_part, presets=PRESETS, time_limit=TIME_LIMIT)

    pos = predictor.positive_class
    oof_pred[va_idx] = predictor.predict_proba(valid_part)[pos].values
    test_pred += predictor.predict_proba(test_sorted[feat_cols])[pos].values / N_SPLITS

    fold_auc = roc_auc_score(data.loc[va_idx, 'gender'], oof_pred[va_idx])
    print(f'[fold {f}] valid AUC = {fold_auc:.5f}')

[fold 0] valid AUC = 0.64106
[fold 1] valid AUC = 0.63624
[fold 2] valid AUC = 0.63943
[fold 3] valid AUC = 0.63354
[fold 4] valid AUC = 0.66036
time: 26min 35s (started: 2026-06-06 20:10:45 +09:00)


## 5. CV 지표

In [36]:
# ===== CV 지표: AUC(순위) + LogLoss/Brier(칼리브레이션·오답 페널티) =====
y_true = data['gender'].values
cv_auc   = roc_auc_score(y_true, oof_pred)
cv_ll    = log_loss(y_true, oof_pred)
cv_brier = brier_score_loss(y_true, oof_pred)

print(f'=== {SETNAME} CV ===')
print(f'AUC      : {cv_auc:.5f}   (순위 품질, 높을수록 좋음)')
print(f'LogLoss  : {cv_ll:.5f}   (확신한 오답에 큰 페널티, 낮을수록 좋음)')
print(f'Brier    : {cv_brier:.5f}   (확률 칼리브레이션, 낮을수록 좋음)')

=== behavior CV ===
AUC      : 0.64163   (순위 품질, 높을수록 좋음)
LogLoss  : 0.58472   (확신한 오답에 큰 페널티, 낮을수록 좋음)
Brier    : 0.19874   (확률 칼리브레이션, 낮을수록 좋음)
time: 27.2 ms (started: 2026-06-06 20:37:20 +09:00)


## 6. OOF / test 예측 저장

In [37]:
# ===== OOF / test 예측 저장 (스키마: custid, pred_proba) =====
oof_out = pd.DataFrame({
    'custid': data['custid'].values,
    'pred_proba': oof_pred,
    'fold': data['fold'].values,
})
oof_out.to_csv(f'{OOF_DIR}/f/oof_{SETNAME}_f.csv', index=False)

test_out = pd.DataFrame({
    'custid': test_sorted['custid'].values,
    'pred_proba': test_pred,
})
test_out.to_csv(f'{OOF_DIR}/f/test_{SETNAME}_f.csv', index=False)

print('저장 완료:')
print(f'  {OOF_DIR}/f/oof_{SETNAME}_f.csv      ', oof_out.shape)
print(f'  {OOF_DIR}/f/test_{SETNAME}_f.csv ', test_out.shape)
display(oof_out.head())

저장 완료:
  ../../outputs/oof/oof_behavior_1.csv       (30000, 3)
  ../../outputs/oof/oof_test_behavior_1.csv  (19995, 2)


,custid,pred_proba,fold
0,0,0.702011,3
1,1,0.303451,1
2,2,0.599130,0
3,3,0.181335,0
4,4,0.297866,0


time: 212 ms (started: 2026-06-06 20:37:20 +09:00)
